# Eurostat EDA
Preparation for Stand-Up meeting on June 2d, 2026

## Configuration:
### Install & Import Exploration Tools
Installing important libraries and `sqlalchemy` (needed to install it into dbt_project conda enviroment as well):

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import numpy as np

# Load your database credentials
load_dotenv(dotenv_path='.env')

### Set the countries of interest (EU)

In [ ]:
eu_countries = (
        'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES', 'FR', 
        'HR', 'IT', 'CY', 'LV', 'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 
        'PL', 'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
)

### Establish the Connection Engine
Instead of basic raw cursors, Pandas uses a SQLAlchemy engine connection to smoothly stream rows straight into tabular DataFrames.

In [ ]:
db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USER")

# Format: postgresql://username@localhost/database_name
engine = create_engine(f"postgresql://{db_user}@localhost/{db_name}")

### Stream a Table straight into a Pandas DataFrame
Now I can read any raw table or execute a direct SQL query right into a DataFrame using `pd.read_sql()`. Example here is about the AI usage table:

In [ ]:
# just the data from one table:
df_survey = pd.read_sql("SELECT * FROM public.isoc_ai_iaiu", con=engine)

In [ ]:
df_survey

In [ ]:
df_survey['indic_is'].unique()

In [ ]:
# or just special individual types (IND_TOTAL) and special unit (PC_IND, percentage of individuals) 
# and for each indicator:
# I_IUAI = usage of AI in the last 3 month
# I_IUAIFE = -''-  for formal education
# I_IUAIPR = -''- for private purposes
# I_IUAIWP = -''- for professional (work) purposes

query = f"""
    SELECT 
        s.*, 
        m.indicator_name 
    FROM public.isoc_ai_iaiu s
    LEFT JOIN public.eurostat_indicators m 
      ON s.indic_is = m.indicator_code
    WHERE s.ind_type = 'IND_TOTAL' 
        and s.indic_is = 'I_IUAI' 
        and s.unit = 'PC_IND'
        and s.country in {eu_countries};
"""
df_combined = pd.read_sql(query, con=engine)

In [ ]:
df_combined.head(3)

In [ ]:
df_combined.describe()

Small viz of AI Usage in the last 3 month by individual in the EU countries (2025):

In [ ]:
df_combined.head(3)

In [ ]:
import matplotlib.pyplot as plt

# 1. Isolate the data series directly
plot_data = df_combined.set_index('country')['year_2025']

# 2. Define colors inline (DE gets purple, everything else gets yellow)
colors = ['#7140AD' if c == 'DE' else '#DEC226' for c in plot_data.index]

# 3. Plot and style everything in one clean block
plt.figure(figsize=(12, 5))
plot_data.plot(kind='bar', color=colors, width=0.7)

plt.title('AI Usage in the EU Countries in the last 3 Months (2025)', fontweight='bold', pad=15)
plt.ylabel('Percentage (%)')
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# or just special individual types (IND_TOTAL) and special unit (PC_IND, percentage of individuals) 
# and for each indicator:
# I_IUAI = usage of AI in the last 3 month
# I_IUAIFE = -''-  for formal education
# I_IUAIPR = -''- for private purposes
# I_IUAIWP = -''- for professional (work) purposes

query_per = f"""
    SELECT 
        s.*, 
        m.indicator_name 
    FROM public.isoc_ai_iaiu s
    LEFT JOIN public.eurostat_indicators m 
      ON s.indic_is = m.indicator_code
    WHERE s.ind_type = 'IND_TOTAL' 
        and s.indic_is = 'I_IUAIPR' 
        and s.unit = 'PC_IND'
        and s.country in {eu_countries};
"""
df_combined_per = pd.read_sql(query_per, con=engine)

In [ ]:
plot_data = df_combined_per.set_index('country')['year_2025']

colors = ['#7140AD' if c == 'DE' else '#DEC226' for c in plot_data.index]

plt.figure(figsize=(12, 5))
plot_data.plot(kind='bar', color=colors, width=0.7)

plt.title('AI Usage in the EU Countries for Personal Purposes', fontweight='bold', pad=15)
plt.ylabel('Percentage (%)')
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# or just special individual types (IND_TOTAL) and special unit (PC_IND, percentage of individuals) 
# and for each indicator:
# I_IUAI = usage of AI in the last 3 month
# I_IUAIFE = -''-  for formal education
# I_IUAIPR = -''- for private purposes
# I_IUAIWP = -''- for professional (work) purposes

query_work = f"""
    SELECT 
        s.*, 
        m.indicator_name 
    FROM public.isoc_ai_iaiu s
    LEFT JOIN public.eurostat_indicators m 
      ON s.indic_is = m.indicator_code
    WHERE s.ind_type = 'IND_TOTAL' 
        and s.indic_is = 'I_IUAIWP' 
        and s.unit = 'PC_IND'
        and s.country in {eu_countries};
"""
df_combined_work = pd.read_sql(query_work, con=engine)

In [ ]:
plot_data = df_combined_work.set_index('country')['year_2025']

colors = ['#7140AD' if c == 'DE' else '#DEC226' for c in plot_data.index]

plt.figure(figsize=(12, 5))
plot_data.plot(kind='bar', color=colors, width=0.7)

plt.title('AI Usage in the EU Countries for Professional Purposes (at work)', fontweight='bold', pad=15)
plt.ylabel('Percentage (%)')
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## Another table: `isoc_ciegi_ac`: E-government activities of individuals via websites

In [ ]:
df_egov = pd.read_sql("SELECT * FROM public.isoc_ciegi_ac", con=engine)

In [ ]:
df_egov.head(10)

In [ ]:
df_egov.shape

In [ ]:
df_egov['indic_is'].unique()

This table is too big so we can work with a selected units, indicators, and individual types:



In [ ]:
query_gov = f"""
    SELECT 
        gov.*, 
        m.indicator_name 
    FROM public.isoc_ciegi_ac gov
    LEFT JOIN public.eurostat_indicators m 
      ON gov.indic_is = m.indicator_code
    WHERE gov.ind_type = 'IND_TOTAL' 
        and gov.indic_is = 'I_IGOV12FM' 
        and gov.unit = 'PC_IND'
        and gov.country in {eu_countries};
"""
df_egov = pd.read_sql(query_work, con=engine)

In [ ]:
df_egov.shape

In [ ]:
df_egov.head()

In [ ]:
plot_data = df_egov.set_index('country')['year_2025']

colors = ['#7140AD' if c == 'DE' else '#DEC226' for c in plot_data.index]

plt.figure(figsize=(12, 5))
plot_data.plot(kind='bar', color=colors, width=0.7)

plt.title('Internet Usage: Downloading Official Forms (last 12 months)', fontweight='bold', pad=15)
plt.ylabel('Percentage (%)')
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## `tin00129`:  Voting

In [ ]:
df_vote = pd.read_sql("SELECT * FROM public.tin00129", con=engine)

In [ ]:
df_vote

In [ ]:
query_vote = f"""
    SELECT 
        tin.*, 
        m.indicator_name 
    FROM public.tin00129 as tin
    LEFT JOIN public.eurostat_indicators m 
      ON gov.indic_is = m.indicator_code
    WHERE tin.ind_type = 'IND_TOTAL' 
        and tin.indic_is = 'I_IUVOTE' 
        and tin.unit = 'PC_IND'
        and tin.country in {eu_countries};
"""
df_vote = pd.read_sql(query_work, con=engine)

In [ ]:
df_vote.shape

In [ ]:
plot_data = df_vote.set_index('country')['year_2025']

colors = ['#7140AD' if c == 'DE' else '#DEC226' for c in plot_data.index]

plt.figure(figsize=(12, 5))
plot_data.plot(kind='bar', color=colors, width=0.7)

plt.title('Internet Usage: Taking part in online consultations or voting to define civic or political issues like signing a petition', fontweight='bold', pad=15)
plt.ylabel('Percentage (%)')
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()